# LAB | Hyperparameter Tuning

**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far we've been training and evaluating models with default values for hyperparameters.

Today we will perform the same feature engineering as before, and then compare the best working models you got so far, but now fine tuning it's hyperparameters.

In [ ]:
# ------------------------------
# LAB | Hyperparameter Tuning
# ------------------------------

# 1️⃣ Librerías
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# 2️⃣ Cargar datos
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")

# 3️⃣ Preprocesamiento
# Eliminar columnas irrelevantes
drop_cols = ['PassengerId', 'Name']
spaceship = spaceship.drop(columns=drop_cols)

# Separar numéricas y categóricas
num_features = ['Age','RoomService','FoodCourt','ShoppingMall','Spa','VRDeck']
cat_features = ['HomePlanet','CryoSleep','Cabin','Destination','VIP']

# Pipelines para numéricas y categóricas
num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', num_transformer, num_features),
    ('cat', cat_transformer, cat_features)
])

# Separar X e y
X = spaceship.drop(columns=['Transported'])
y = spaceship['Transported'].astype(int)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4️⃣ Pipeline con Random Forest
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

# 5️⃣ Definir Grid de hiperparámetros
param_grid = {
    'classifier__n_estimators': [100, 200, 500],
    'classifier__max_depth': [None, 10, 20, 30],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__min_samples_leaf': [1, 2, 4],
    'classifier__max_features': ['sqrt', 'log2', None]
}

# 6️⃣ Grid Search
grid_search = GridSearchCV(
    rf_pipeline,
    param_grid,
    cv=5,
    n_jobs=-1,
    scoring='accuracy',
    verbose=1
)

grid_search.fit(X_train, y_train)

# 7️⃣ Evaluación del mejor modelo
print("Best parameters found:", grid_search.best_params_)
best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)
print("Test Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=["Not Transported", "Transported"]))



Fitting 5 folds for each of 324 candidates, totalling 1620 fits


Now perform the same as before:
- Feature Scaling
- Feature Selection


- Now let's use the best model we got so far in order to see how it can improve when we fine tune it's hyperparameters.

**Grid/Random Search**

For this lab we will use Grid Search.

- Define hyperparameters to fine tune.

- Evaluate your model